In [0]:
import pandas as pd
from pyspark.sql.functions import col, lit,  when

#Global

In [0]:
tb_moni = 'ia.tb_diamond_mod_monitoramento'

tb_entrada = 'ia.tb_diamond_creatinina_wrk_01'
tb_saida = 'ia.tb_diamond_nefrologia_clearance_saida'

#Criacao da tabela de monitoramento

In [0]:
df_entrada = spark.sql(f"""
    select 
        dt_execucao,
        org.sigEstado  as nm_regional,
        COUNT(id_exame) AS qtd_entrada,
        nme_unidade as nm_unidade
    from {tb_entrada} as wrk
    left join (select `_id` as id, 
                  documento.numCNPJ,
                  endereco.sigEstado
          from gold_corporativo.organization.tb_gold_mov_organization 
          qualify row_number () over (partition by `_id` order by tmpDataAtualizacao desc) = 1 
          )
          as org 
  on wrk.id_unidade = id
  where org.sigEstado is not null
    GROUP BY dt_execucao,org.sigEstado,nme_unidade
    ORDER BY dt_execucao
""")

df_saida = spark.sql(f"""
    SELECT
        dt_execucao,
        regionalUnidade as nm_regional,
        COUNT(idExame) AS qtd_saida,
        nomeUnidade as nm_unidade
    FROM {tb_saida}
    where dt_execucao is not null
        and cast(valorClearance as numeric) between 0.1 and 60
        and nomeConvenio  not rlike r'(?i)(?<!\w)AMIL(?:\s*[-/().A-Za-z0-9]+)?' 
        -- and cast(dataExameRecente as date) <= current_date()
        and regionalUnidade is not null
    GROUP BY dt_execucao,regionalUnidade,nomeUnidade
    ORDER BY dt_execucao
""")

df_relevante = spark.sql(f"""
    SELECT
        dt_execucao,
        regionalUnidade as nm_regional,
        COUNT(idExame) AS qtd_relevante,
        nomeUnidade as nm_unidade
    FROM {tb_saida}
    where dt_execucao is not null
        and cast(valorClearance as numeric) between 0.1 and 35
        and nomeConvenio  not rlike r'(?i)(?<!\w)AMIL(?:\s*[-/().A-Za-z0-9]+)?' 
        -- and cast(dataExameRecente as date) <= current_date()
        and regionalUnidade is not null
    GROUP BY dt_execucao,regionalUnidade,nomeUnidade
    ORDER BY dt_execucao
""")

In [0]:
from pyspark.sql.functions import current_date

df_entrada_filtrado = df_entrada.filter(
    df_entrada["dt_execucao"] == current_date()
)

display(df_entrada_filtrado)

In [0]:
df_saida_filtrado = df_saida.filter(
    df_saida["dt_execucao"] == current_date()
)

display(df_saida_filtrado)

In [0]:
df_relevante_filtrado = df_relevante.filter(
    df_relevante["dt_execucao"] == current_date()
)

display(df_relevante_filtrado)

In [0]:
df_monitoramento = (
    df_entrada.alias("e")
        .join(df_saida.alias("s"), on=["dt_execucao", "nm_regional","nm_unidade"], how="left")
        .join(df_relevante.alias("r"), on=["dt_execucao", "nm_regional","nm_unidade"], how="left")
        .select(
            "dt_execucao",
            "e.qtd_entrada",
            "s.qtd_saida",
            "r.qtd_relevante",
            "e.nm_regional", 
            "e.nm_unidade"
        )
)

In [0]:
df_monitoramento_filtrado = df_monitoramento.filter(
    df_monitoramento["dt_execucao"] == current_date()
)

display(df_monitoramento_filtrado)

In [0]:
df_monitoramento = (
    df_monitoramento
        .withColumn("qtd_saida", col("qtd_saida").cast("int"))
        .withColumn("qtd_entrada", col("qtd_entrada").cast("int"))
        .withColumn("qtd_relevante", col("qtd_relevante").cast("int"))
        .withColumn("nm_regional", col("nm_regional").cast("string"))
        .withColumn("nm_unidade", col("nm_unidade").cast("string"))
        .withColumn("nm_projeto", lit("Rim").cast("string"))
)

In [0]:
df_monitoramento.createOrReplaceTempView("vw_monitoramento_wrk_01")


In [0]:
%sql
select * from vw_monitoramento_wrk_01 where dt_execucao = current_date()

## Inclusão da volumetria histórica de dados relevantes

In [0]:
# df_relevante = (
#     df_relevante
#         .withColumn("qtd_relevante", col("qtd_relevante").cast("int"))
#         .withColumn("nm_regional", col("nm_regional").cast("string"))
#         .withColumn("nm_unidade", col("nm_unidade").cast("string"))
#         .withColumn("nm_projeto", lit("Rim").cast("string"))
# )
# df_relevante.createOrReplaceTempView("vw_monitoramento_wrk_02")

In [0]:
# %sql
# MERGE INTO ia.tb_diamond_mod_monitoramento t
# USING vw_monitoramento_wrk_02 s
# ON  t.dt_execucao = s.dt_execucao
# AND t.nm_regional = s.nm_regional
# AND t.nm_unidade = s.nm_unidade
# AND t.nm_projeto  = s.nm_projeto

# WHEN MATCHED THEN
#   UPDATE SET
#     t.qtd_relevante = s.qtd_relevante;

In [0]:
%sql
MERGE INTO ia.tb_diamond_mod_monitoramento t
USING vw_monitoramento_wrk_01 s
ON  t.dt_execucao = s.dt_execucao
AND t.nm_regional = s.nm_regional
AND t.nm_unidade = s.nm_unidade
AND t.nm_projeto  = s.nm_projeto

WHEN NOT MATCHED THEN
  INSERT (
    dt_execucao,
    qtd_entrada,
    qtd_saida,
    qtd_relevante,
    nm_regional,
    nm_unidade,
    nm_projeto
    
  )
  VALUES (
    s.dt_execucao,
    s.qtd_entrada,
    s.qtd_saida,
    s.qtd_relevante,
    s.nm_regional,
    s.nm_unidade,
    s.nm_projeto
  )


In [0]:
# Use esse bloco apenas se a tabela de monitoramento estiver vazia(Drop Table)
# df_monitoramento.write.mode("append").saveAsTable(tbl_moni)

In [0]:
dbutils.notebook().exit("Fim da execução!")

In [0]:
%sql
select 
dt_execucao,
qtd_entrada,
qtd_saida,
qtd_relevante,
nm_regional,
nm_unidade
from ia.tb_diamond_mod_monitoramento 
where nm_projeto = "Rim" 
and nm_unidade is not null
--and nm_regional = "RJ" 
--and dt_execucao = "2026-02-05"